# Solutions 06: Sequence-Level and Black-Box Distillation

This notebook solves the four exercises of Lab 06 (`labs/lab-06-seqkd-and-blackbox.ipynb`).
Execution status: every exercise has a live, executed component; exercises 1, 2, and 3 also
carry a full training or regeneration run that is written out but gated behind
`RUN_TRAINING = False`, exactly like the lab's own Part B. Exercise 4 is arithmetic and runs
entirely live.

Attempt the exercises yourself before reading this. The point of an exercise is the hour you
spend stuck on it; the solution only certifies where you landed.

In [1]:
import os
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")   # widget progress bars crash some notebook stacks; plain logs are fine

import sys, os, json, math, time, inspect
sys.path.insert(0, "../code")
os.environ["TRL_EXPERIMENTAL_SILENCE"] = "1"

import torch
import torch.nn.functional as F

from kd_core import (completion_mask_from_prompt_lens, onpolicy_mask,
                     distinct_n, self_bleu)
from kd_pipeline import (set_seed_everywhere, config_fingerprint,
                         infer_gb, full_ft_gb, bandwidth_bound_decode_tps,
                         decode_wallclock_hours, prefill_wallclock_hours, RunManifest)

RUN_TRAINING = False        # <-- flip on the training box
SEED = 17
set_seed_everywhere(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {device} | RUN_TRAINING: {RUN_TRAINING}")

torch 2.13.0+cpu | device: cpu | RUN_TRAINING: False


## Exercise 1: Sampled vs greedy SeqKD

**The exercise, restated.** Regenerate the SeqKD corpus at temperature T=1.0 (sampling from
the teacher's full next-token distribution) instead of greedy decoding (always taking the
single most likely token), retrain the student on it, and ask: which wins, on which metric,
and does the entropy gap against the cached-logit student close?

**The approach.** The full regenerate-plus-retrain comparison costs the priced decode phase
from the lab's Part A·1 twice, so that part goes behind the flag. But the *mechanism* the
exercise is about, that greedy and sampled decoding produce data sources with different
diversity, does not need a 1.7B teacher or any training to see. It is a property of the
generated text itself, and a 135M model generating a handful of completions shows it in two
ways:

1. *Within one prompt:* greedy decoding is deterministic, which means asking the same model
   the same prompt four times yields four byte-identical completions. Sampling at T=1.0
   yields four different ones. This is the entire difference between the two data sources in
   miniature: a greedy corpus contains exactly one continuation per prompt, the teacher's
   mode, while a sampled corpus is a draw from the teacher's whole distribution over
   continuations.
2. *Across prompts:* the same diversity metrics the lab's Part C uses, distinct-3 (unique
   3-grams divided by total 3-grams, higher means more varied text) and self-BLEU (how much
   each completion's 3-grams overlap the others', higher means the completions resemble each
   other), computed on the greedy set and the sampled set.

The live cells below run that demonstration on 8 prompts from the shared eval corpus with
SmolLM2-135M in fp32, seeded. The gated cell then gives the full recipe: the lab's own
`generate_corpus` with `greedy=False`, the same `sft` loop, and the Part C evaluation.

In [2]:
# Live: generate greedy and sampled completions from a small model on real prompts.
from transformers import AutoModelForCausalLM, AutoTokenizer

GEN_MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"     # small on purpose: mechanism demo
tok = AutoTokenizer.from_pretrained(GEN_MODEL)
model = AutoModelForCausalLM.from_pretrained(GEN_MODEL, dtype=torch.float32).eval()

ev = torch.load("../data/lab03/eval.pt")
N_PROMPTS, MAX_NEW = 8, 32
plens = [int(ev["prompt_lens"][i]) for i in range(N_PROMPTS)]

# Left-pad the prompts into one batch so a single generate call serves all of them.
# Left padding (pad tokens before the prompt, not after) is required for batched
# generation because the model continues from the last position of each row.
Lmax = max(plens)
batch = torch.full((N_PROMPTS, Lmax), tok.eos_token_id, dtype=torch.long)
attn = torch.zeros((N_PROMPTS, Lmax), dtype=torch.long)
for i, pl in enumerate(plens):
    batch[i, Lmax - pl:] = ev["input_ids"][i, :pl]
    attn[i, Lmax - pl:] = 1

def decode_new(gen):
    texts = []
    for row in gen[:, Lmax:]:
        ids = row.tolist()
        if tok.eos_token_id in ids:
            ids = ids[:ids.index(tok.eos_token_id)]
        texts.append(tok.decode(ids, skip_special_tokens=True))
    return texts

t0 = time.time()
set_seed_everywhere(SEED)
with torch.no_grad():
    g_greedy = model.generate(input_ids=batch, attention_mask=attn, do_sample=False,
                              max_new_tokens=MAX_NEW, pad_token_id=tok.eos_token_id)
set_seed_everywhere(SEED)
with torch.no_grad():
    g_sample = model.generate(input_ids=batch, attention_mask=attn, do_sample=True,
                              temperature=1.0, max_new_tokens=MAX_NEW,
                              pad_token_id=tok.eos_token_id)
greedy_texts, sampled_texts = decode_new(g_greedy), decode_new(g_sample)
print(f"generated 2 x {N_PROMPTS} completions of up to {MAX_NEW} tokens "
      f"in {time.time()-t0:.0f}s")
print("greedy  [0]:", greedy_texts[0][:100] + "...")
print("sampled [0]:", sampled_texts[0][:100] + "...")

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/272 [00:00<00:28,  9.43it/s]

Loading weights:  42%|████▏     | 113/272 [00:00<00:00, 645.74it/s]

Loading weights:  67%|██████▋   | 183/272 [00:00<00:00, 665.80it/s]

Loading weights:  95%|█████████▌| 259/272 [00:00<00:00, 699.37it/s]

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 643.28it/s]

generated 2 x 8 completions of up to 32 tokens in 124s
greedy  [0]: A person's favorite food is a topic of discussion in the given paragraph....
sampled [0]: I love to eat, people's favorite food....


In [3]:
# Live: the within-prompt determinism demo, then the diversity metrics.
K_REP = 4
rep = batch[0:1].repeat(K_REP, 1)
rep_attn = attn[0:1].repeat(K_REP, 1)
set_seed_everywhere(SEED)
with torch.no_grad():
    r_greedy = model.generate(input_ids=rep, attention_mask=rep_attn, do_sample=False,
                              max_new_tokens=MAX_NEW, pad_token_id=tok.eos_token_id)
set_seed_everywhere(SEED)
with torch.no_grad():
    r_sample = model.generate(input_ids=rep, attention_mask=rep_attn, do_sample=True,
                              temperature=1.0, max_new_tokens=MAX_NEW,
                              pad_token_id=tok.eos_token_id)
within_greedy, within_sampled = decode_new(r_greedy), decode_new(r_sample)

n_unique_greedy = len(set(within_greedy))
n_unique_sampled = len(set(within_sampled))
sb_within_greedy = self_bleu(within_greedy, n=3)
sb_within_sampled = self_bleu(within_sampled, n=3)

def toklen(texts):
    return sum(len(t.split()) for t in texts) / max(1, len(texts))

print("within one prompt, 4 generations each:")
print(f"  greedy : {n_unique_greedy} unique completion(s), self-BLEU {sb_within_greedy:.3f}")
print(f"  sampled: {n_unique_sampled} unique completion(s), self-BLEU {sb_within_sampled:.3f}")
print()
print("across the 8 prompts:")
print(f"  {'arm':>8} {'distinct-3':>11} {'self-BLEU-3':>12} {'mean words':>11}")
for name, texts in (("greedy", greedy_texts), ("sampled", sampled_texts)):
    print(f"  {name:>8} {distinct_n(texts, 3):>11.3f} "
          f"{self_bleu(texts, 3):>12.3f} {toklen(texts):>11.1f}")

# The load-bearing checks: greedy is one continuation per prompt, sampling is many.
assert n_unique_greedy == 1, "greedy decoding must be deterministic"
assert n_unique_sampled >= 3, "T=1.0 sampling must produce distinct continuations"
assert sb_within_greedy >= 0.999, "identical texts must have self-BLEU of 1"
assert sb_within_sampled < sb_within_greedy, \
    "sampled completions must overlap each other less than identical ones"
print("\nchecks passed: greedy is a one-point data source per prompt, sampling is not")

within one prompt, 4 generations each:
  greedy : 1 unique completion(s), self-BLEU 1.000
  sampled: 4 unique completion(s), self-BLEU 0.174

across the 8 prompts:
       arm  distinct-3  self-BLEU-3  mean words
    greedy       0.919        0.060        23.6
   sampled       0.953        0.000        23.5

checks passed: greedy is a one-point data source per prompt, sampling is not


In [4]:
# Gated: the full regenerate-and-retrain comparison, exactly the lab's Part B
# with greedy=False, then the lab's Part C metrics on both students.
CFG = dict(
    gen_teacher="HuggingFaceTB/SmolLM2-1.7B-Instruct",
    student="HuggingFaceTB/SmolLM2-360M-Instruct",
    data="../data/lab03", seq_len=384, gen_max_new=256, n_gen_prompts=2048,
    lr=3e-5, batch_size=8, grad_accum=4, max_steps=1500, warmup_steps=50,
)

if RUN_TRAINING:
    from transformers import get_cosine_schedule_with_warmup

    def generate_corpus(cfg, greedy):
        gtok = AutoTokenizer.from_pretrained(cfg["gen_teacher"])
        teacher = AutoModelForCausalLM.from_pretrained(
            cfg["gen_teacher"], dtype=torch.bfloat16).to(device).eval()
        tr = torch.load(os.path.join(cfg["data"], "train.pt"))
        outs, dropped = [], 0
        for i in range(cfg["n_gen_prompts"]):
            plen = tr["prompt_lens"][i]
            prompt = tr["input_ids"][i:i+1, :plen].to(device)
            with torch.no_grad():
                gen = teacher.generate(
                    prompt, do_sample=not greedy,
                    temperature=None if greedy else 1.0,
                    max_new_tokens=cfg["gen_max_new"], pad_token_id=gtok.eos_token_id)
            comp = gen[0, plen:]
            if gtok.eos_token_id not in comp:
                dropped += 1
                continue
            outs.append({"prompt_len": int(plen), "ids": gen[0].cpu()})
        del teacher
        print(f"greedy={greedy}: kept {len(outs)}, dropped {dropped} capped generations")
        return outs

    def pack(outs, cfg, stok):
        pad = stok.eos_token_id
        ids = torch.full((len(outs), cfg["seq_len"]), pad, dtype=torch.long)
        pls = []
        for i, o in enumerate(outs):
            seq = o["ids"][:cfg["seq_len"]]
            ids[i, :len(seq)] = seq
            pls.append(o["prompt_len"])
        mask = completion_mask_from_prompt_lens(ids, pls, pad_token_id=pad)
        labels = ids.clone(); labels[~mask] = -100
        return ids, labels, mask

    def sft(name, ids, labels, cfg):
        set_seed_everywhere(SEED)
        student = AutoModelForCausalLM.from_pretrained(
            cfg["student"], dtype=torch.bfloat16).to(device)
        opt = torch.optim.AdamW(student.parameters(), lr=cfg["lr"])
        sched = get_cosine_schedule_with_warmup(opt, cfg["warmup_steps"], cfg["max_steps"])
        step = 0
        while step < cfg["max_steps"]:
            for i in range(0, len(ids), cfg["batch_size"]):
                b_ids = ids[i:i+cfg["batch_size"]].to(device)
                b_lab = labels[i:i+cfg["batch_size"]].to(device)
                logits = student(b_ids).logits
                loss = F.cross_entropy(
                    logits[:, :-1].reshape(-1, logits.shape[-1]),
                    b_lab[:, 1:].reshape(-1), ignore_index=-100)
                (loss / cfg["grad_accum"]).backward()
                if (step + 1) % cfg["grad_accum"] == 0:
                    torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
                    opt.step(); sched.step(); opt.zero_grad()
                step += 1
                if step >= cfg["max_steps"]:
                    break
        out = f"../runs/lab06/{name}_{config_fingerprint({**cfg, 'seed': SEED, 'arm': name})}"
        os.makedirs(out, exist_ok=True)
        student.save_pretrained(out)
        del student
        return out

    stok = AutoTokenizer.from_pretrained(CFG["student"])
    outs_T1 = generate_corpus(CFG, greedy=False)          # the priced decode, again
    ids, labels, mask = pack(outs_T1, CFG, stok)
    ckpt_T1 = sft("seqkd-T1", ids, labels, CFG)
    print("sampled-SeqKD checkpoint:", ckpt_T1)
    # Then rerun the lab's Part C metrics (agreement, KL, entropy, distinct-3)
    # on seqkd-greedy, seqkd-T1, and the Lab 04 cached student.
else:
    print("RUN_TRAINING=False: the regenerate-and-retrain arm compiled but did not run.")

RUN_TRAINING=False: the regenerate-and-retrain arm compiled but did not run.


**Interpretation.** The live cells show the mechanism the exercise turns on. Greedy decoding
gave exactly 1 unique completion out of 4 tries on the same prompt with a within-set
self-BLEU of 1.000, while T=1.0 sampling gave several distinct completions with a lower
self-BLEU, and the across-prompt table shows the same contrast in distinct-3. So a greedy
corpus and a sampled corpus are genuinely different training distributions before any
training happens: the greedy corpus is one point per prompt (the teacher's mode), the
sampled corpus is a spread of draws per prompt.

The retraining itself did not execute here (one line of honesty: `RUN_TRAINING=False`, so
the gated cell compiled only). Expected outcome, grounded in the lab's Part C ranges: the
sampled-SeqKD student's entropy should land 0.1 to 0.3 nats *above* the greedy-SeqKD
student's, closing part of the 0.1 to 0.4 nat gap Part C predicts against the cached-logit
student, because the data source now carries some of the teacher's spread instead of only
its mode. On top-1 agreement, expect greedy to hold a small edge (a point or two): its data
concentrates supervision on exactly the tokens agreement measures. That is the trade the
exercise wants named: sampling buys back diversity and calibration at a small cost in
mode-matching, so "which wins" depends on whether your deployment metric rewards the mode
(greedy wins) or the distribution (sampling wins). Confirming evidence would be the entropy
gap closing while agreement moves at most a point or two; refuting evidence would be sampled
entropy *not* rising, which would say the student, not the data, is the diversity
bottleneck. Failure signature to watch: a high capped-generation drop rate at T=1.0, since
sampling wanders and hits the length cap more often than greedy does, and the drop rate is a
data-quality number, not noise.

## Exercise 2: The `seq_kd=True` cross-check

**The exercise, restated.** Run TRL's GKD trainer with `seq_kd=True, lmbda=1.0` on the same
prompts as the hand-rolled SeqKD arm and compare. They should land close; where they differ,
read the trainer source to find the decision made differently.

**The approach.** The lab's own standing rule is to introspect the installed object rather
than trust documentation or memory, so before spending a training run I read the installed
`GKDTrainer.training_step` source and the `GKDConfig` fields. That reading settles the
exercise's configuration *before* any compute is spent, and it turns out to matter more than
expected: the branch structure of `training_step` decides whether `seq_kd=True` does
anything at all, and the exercise's suggested `lmbda=1.0` interacts with it. The live cell
prints the exact lines and asserts the branch structure; the run itself is gated.

In [5]:
# Live: introspect the installed library path for SeqKD.
import dataclasses
from trl.experimental.gkd import GKDConfig
import trl.experimental.gkd.gkd_trainer as gkd_trainer_module
from trl.experimental.gkd import GKDTrainer

fields = {f.name: f.default for f in dataclasses.fields(GKDConfig)}
for k in ("seq_kd", "lmbda", "beta", "temperature", "max_new_tokens"):
    assert k in fields, f"TRL drift: GKDConfig lost {k}"
    print(f"GKDConfig.{k:<16} default = {fields[k]}")

src = inspect.getsource(GKDTrainer.training_step)
srcfile = inspect.getsourcefile(gkd_trainer_module)
first_line = inspect.getsourcelines(GKDTrainer.training_step)[1]
print(f"\ntraining_step source: {srcfile}, starting line {first_line}")
print("the two branches that decide who generates:")
for off, line in enumerate(src.splitlines()):
    s = line.strip()
    if ("random.random() <= self.lmbda" in s or "elif self.seq_kd" in s
            or "generate_on_policy_outputs" in s or "self.teacher_model," in s):
        print(f"  L{first_line + off:>4}: {line.rstrip()}")

i_lmbda = src.index("random.random() <= self.lmbda")
i_seqkd = src.index("elif self.seq_kd")
assert i_lmbda < i_seqkd, "lmbda branch must come first for the elif to mean what it means"
assert "elif self.seq_kd:" in src, "seq_kd is an elif, reachable only when the lmbda draw fails"
print("\nverified: per step, a uniform draw against lmbda picks STUDENT generation;")
print("only when that draw fails does seq_kd make the TEACHER generate.")
print("So seq_kd=True with lmbda=1.0 NEVER takes the SeqKD branch: every step is")
print("on-policy GKD. Library SeqKD is seq_kd=True with lmbda=0.0.")

GKDConfig.seq_kd           default = False
GKDConfig.lmbda            default = 0.5
GKDConfig.beta             default = 0.5
GKDConfig.temperature      default = 0.9
GKDConfig.max_new_tokens   default = 128

training_step source: /usr/local/lib/python3.11/dist-packages/trl/experimental/gkd/gkd_trainer.py, starting line 491
the two branches that decide who generates:
  L 501:         if random.random() <= self.lmbda:
  L 509:                 new_input_ids, new_attention_mask, new_labels = self.generate_on_policy_outputs(
  L 515:         elif self.seq_kd:
  L 518:                     self.teacher_model,
  L 523:                 new_input_ids, new_attention_mask, new_labels = self.generate_on_policy_outputs(

verified: per step, a uniform draw against lmbda picks STUDENT generation;
only when that draw fails does seq_kd make the TEACHER generate.
So seq_kd=True with lmbda=1.0 NEVER takes the SeqKD branch: every step is
on-policy GKD. Library SeqKD is seq_kd=True with lmbda=0.0.


In [6]:
# Gated: the corrected library run, seq_kd=True and lmbda=0.0.
if RUN_TRAINING:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from trl.experimental.gkd import GKDTrainer
    from datasets import Dataset

    stok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct")
    student = AutoModelForCausalLM.from_pretrained(
        "HuggingFaceTB/SmolLM2-360M-Instruct", dtype=torch.bfloat16)
    teacher = AutoModelForCausalLM.from_pretrained(
        "HuggingFaceTB/SmolLM2-1.7B-Instruct", dtype=torch.bfloat16)
    tr = torch.load("../data/lab03/train.pt")
    msgs = []
    for i in range(1024):
        text = stok.decode(tr["input_ids"][i, :tr["prompt_lens"][i]],
                           skip_special_tokens=True)
        msgs.append({"messages": [{"role": "user", "content": text},
                                  {"role": "assistant", "content": ""}]})
    args = GKDConfig(
        output_dir="../runs/lab06/trl-seqkd",
        seq_kd=True, lmbda=0.0,            # the introspection-corrected setting
        beta=0.5, temperature=1.0, max_new_tokens=256,
        per_device_train_batch_size=4, gradient_accumulation_steps=8,
        learning_rate=3e-5, max_steps=600, logging_steps=25, bf16=True,
        report_to=[],
    )
    trainer = GKDTrainer(model=student, teacher_model=teacher, args=args,
                         processing_class=stok,
                         train_dataset=Dataset.from_list(msgs))
    trainer.train()
    trainer.save_model("../runs/lab06/trl-seqkd/final")
else:
    print("RUN_TRAINING=False: the library SeqKD run compiled but did not execute.")

RUN_TRAINING=False: the library SeqKD run compiled but did not execute.


**Interpretation.** The exercise's premise needed a correction, and the introspection found
it without spending a single training step: in the installed TRL, `training_step` draws a
uniform random number each step and takes the student-generates branch whenever the draw is
at or below `lmbda`; the teacher-generates SeqKD branch is an `elif` behind that draw. With
`lmbda=1.0` the draw always succeeds, so `seq_kd=True` is dead code and the run silently
becomes pure on-policy GKD, a different experiment producing a perfectly normal loss curve.
The library equivalent of the lab's hand-rolled SeqKD arm is `seq_kd=True, lmbda=0.0`. This
is the course's documentation-drift lesson landing on its own exercise text, and reporting
it beats forcing the expected answer.

The corrected run is gated (`RUN_TRAINING=False`; the cell compiled only). Expected result,
grounded in Part C: the library arm should land within noise of the hand-rolled arm, meaning
agreement within a point or two and entropy within about 0.1 nats, because both are
cross-entropy on teacher generations for the same prompts. The differences to hunt for when
they diverge, each visible in the printed source path: the trainer regenerates teacher
completions *every step* at `temperature` (default 0.9), where the hand-rolled arm generated
one greedy corpus once, so the library arm is closer to Exercise 1's sampled variant than to
classical mode-SeqKD; the trainer masks everything after the first EOS but does not *drop*
capped EOS-less generations the way `generate_corpus` does; and prompts pass through the
chat template rather than raw token ids. A large gap in entropy between the two arms most
likely traces to the first difference, sampled-per-step data versus a fixed greedy corpus,
not to a bug.

## Exercise 3: Rationale distillation

**The exercise, restated.** Prepend teacher-generated step-by-step rationales (the worked
reasoning a model writes before its answer) to the completions and run SFT on the combined
text. Does the student improve on held-out prompts *without* rationales at inference time?

**The approach.** Two things can be done live without a training run: build the exact data
format the rationale arm would train on, so the format decision is concrete rather than
hand-waved, and price it. Pricing matters here because rationales are pure decode: every
rationale token is one more token the 1.7B teacher must generate, and decode is the
bandwidth-bound phase where each generated token streams all teacher weights through memory
once. The live cell measures real completion lengths from the shared corpus, measures a
realistic rationale's token count with the actual tokenizer, and prices plain SeqKD against
rationale SeqKD with the lab's own cost model (`bandwidth_bound_decode_tps` at the course's
273 GB/s figure, then `decode_wallclock_hours`). The identity to assert: generation cost
scales exactly with tokens generated, so the cost ratio must equal the token ratio. The
training run and the rationale-free evaluation are gated.

In [7]:
# Live: the rationale-prepended format, and what it costs.
from transformers import AutoTokenizer
stok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct")

# The format. The rationale is generated by the teacher under a think-step-by-step
# instruction, then spliced before the final answer inside the assistant turn.
example_prompt = "Rewrite this sentence in the passive voice: The cat chased the mouse."
example_rationale = ("Reasoning: the task asks for passive voice, which means the object "
                     "of the active sentence becomes the subject. The object here is the "
                     "mouse, the verb is chased, so the passive form uses was chased by. "
                     "I keep the tense and drop nothing.")
example_answer = "The mouse was chased by the cat."
rationale_turn = example_rationale + "\nAnswer: " + example_answer

chat = stok.apply_chat_template(
    [{"role": "user", "content": example_prompt},
     {"role": "assistant", "content": rationale_turn}], tokenize=False)
print("the training text the rationale arm would see:")
print(chat[:400])

# Token accounting on the real corpus.
tr = torch.load("../data/lab03/train.pt")
mean_comp = float(tr["mask"].sum(1).float().mean())          # completion tokens per row
rat_tokens = len(stok(example_rationale + "\nAnswer: ")["input_ids"])
N = 2048                                                     # the lab's corpus size
plain_tokens = N * mean_comp
rationale_tokens = N * (mean_comp + rat_tokens)

tps_17 = bandwidth_bound_decode_tps(1.7, 273.0)              # 1.7B teacher decode ceiling
h_plain = decode_wallclock_hours(plain_tokens, tps_17)
h_rat = decode_wallclock_hours(rationale_tokens, tps_17)
print(f"\nmean completion length          : {mean_comp:8.1f} tokens")
print(f"rationale overhead (measured)   : {rat_tokens:8d} tokens per example")
print(f"plain corpus decode             : {plain_tokens/1e6:8.2f} M tokens "
      f"-> {h_plain:5.2f} single-stream hours")
print(f"rationale corpus decode         : {rationale_tokens/1e6:8.2f} M tokens "
      f"-> {h_rat:5.2f} single-stream hours")
print(f"cost multiplier                 : {h_rat/h_plain:8.2f}x")

assert rat_tokens > 0 and rationale_tokens > plain_tokens
assert abs((h_rat / h_plain) - (rationale_tokens / plain_tokens)) < 1e-9, \
    "decode cost must scale exactly with tokens generated"
print("\nidentity holds: cost ratio == token ratio; rationales are priced in decode tokens")

the training text the rationale arm would see:
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Rewrite this sentence in the passive voice: The cat chased the mouse.<|im_end|>
<|im_start|>assistant
Reasoning: the task asks for passive voice, which means the object of the active sentence becomes the subject. The object here is the mouse, the verb is chased, so the passive form us

mean completion length          :    115.1 tokens
rationale overhead (measured)   :       55 tokens per example
plain corpus decode             :     0.24 M tokens ->  0.82 single-stream hours
rationale corpus decode         :     0.35 M tokens ->  1.21 single-stream hours
cost multiplier                 :     1.48x

identity holds: cost ratio == token ratio; rationales are priced in decode tokens


In [8]:
# Gated: generate rationales with the teacher, SFT, evaluate WITHOUT rationales.
if RUN_TRAINING:
    from transformers import AutoModelForCausalLM
    teacher_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
    gtok = AutoTokenizer.from_pretrained(teacher_name)
    teacher = AutoModelForCausalLM.from_pretrained(
        teacher_name, dtype=torch.bfloat16).to(device).eval()
    tr = torch.load("../data/lab03/train.pt")
    THINK = "Think step by step, then give the final answer after the word Answer:."
    corpus = []
    for i in range(2048):
        plen = tr["prompt_lens"][i]
        user_text = gtok.decode(tr["input_ids"][i, :plen], skip_special_tokens=True)
        msgs = [{"role": "user", "content": user_text + "\n" + THINK}]
        enc = gtok.apply_chat_template(msgs, return_tensors="pt",
                                       add_generation_prompt=True).to(device)
        with torch.no_grad():
            gen = teacher.generate(enc, do_sample=False, max_new_tokens=384,
                                   pad_token_id=gtok.eos_token_id)
        if gtok.eos_token_id not in gen[0, enc.shape[1]:]:
            continue                       # capped: no ending, drop it
        corpus.append({"prompt": user_text,
                       "rationale_completion": gtok.decode(
                           gen[0, enc.shape[1]:], skip_special_tokens=True)})
    del teacher
    torch.save(corpus, "../data/lab06_rationale_corpus.pt")
    # Then: pack WITHOUT the think instruction in the prompt (the student must not
    # need it at inference), SFT with the exercise-1 sft() loop, and evaluate on
    # eval.pt prompts as-is, scoring only the text after "Answer:".
else:
    print("RUN_TRAINING=False: rationale generation and SFT compiled but did not run.")

RUN_TRAINING=False: rationale generation and SFT compiled but did not run.


**Interpretation.** The live accounting settles the economics: with the measured mean
completion length and a realistic rationale measured at its true token count with the actual
tokenizer, the rationale corpus costs the printed multiplier (roughly 1.3x to 1.6x for a 40
to 80 token rationale against this corpus's completions) in teacher decode hours, and the
assert confirms the cost ratio is exactly the token ratio, because on bandwidth-bound decode
every generated token pays the same weight-streaming price. So rationale distillation is not
a free trick: it is a decision to spend 30 to 60 percent more of the most expensive phase.

The training and evaluation are gated (`RUN_TRAINING=False`; nothing beyond the format and
pricing executed). What would count as confirmation: the rationale-trained student beating
plain SeqKD on held-out prompts *served without any rationale request*, since the exercise's
question is whether reasoning practice during training transfers to plain inference. The
published pattern this imitates (distilling reasoning traces into small models) suggests the
gain concentrates on prompts that need multi-step structure and can be zero elsewhere. What
would refute the premise: the student improving only when it is *allowed* to emit the
rationale first, which would mean it learned the format, not the capability, the lab's
style-transfer-without-capability-transfer failure. The failure signature to check first is
formatting leakage: a student that starts every answer with "Reasoning:" on prompts that
asked for none has been taught the scaffold as content; the fix is masking the rationale
tokens out of the loss (supervise only the answer span) and paying the rationale cost purely
as context, which is a second, cheaper arm worth running for the comparison.

## Exercise 4: The matched-compute referee

**The exercise, restated.** Part C compares the three students at matched student *steps*.
Recompute the table at matched *total* compute, teacher work plus student work, and ask
whether the ranking changes.

**The approach.** This is arithmetic, so it runs fully live. The course's cost currency on
bandwidth-limited hardware is bytes of weights streamed from memory, because time equals
bytes moved divided by bandwidth (273 GB/s here). I price each arm's teacher phase and
student phase in those terms:

- *Teacher decode* (the seqkd arm): every generated token streams all 3.4 GB of the 1.7B
  teacher's bf16 weights (1.7 billion parameters times 2 bytes each); batching 16 sequences
  amortizes one weight read across 16 tokens.
- *Teacher prefill* (the cached arm): one weight read serves an entire batch of already
  existing sequences, so the traffic is weight bytes times the number of batches, which is
  tiny.
- *Trace arm:* zero teacher work; the corpus was purchased.
- *Student training:* per optimizer step, the traffic is roughly the full fine-tuning
  working set (the lab's 16 bytes per parameter: weights, gradients, and optimizer moments)
  streamed about three times (forward, backward, update) for the 0.36B student.

This model ignores activations and attention math on both sides, which is stated, and fine
for a referee: those costs hit all arms in the same direction. Then the referee move: set
the budget to the most expensive arm's total, hand every cheaper arm the difference as extra
student steps, and see whether plausible per-step quality gains from Part C's ranges could
flip the ranking. Three identities get asserted: seconds equal bytes over bandwidth, totals
match the budget after reallocation, and the extra-step ordering is trace, then cached, then
seqkd.

In [9]:
# Live: the matched-compute accounting, entirely in bytes and seconds.
BW = 273.0                                  # GB/s, the course's bandwidth figure
T_GB = infer_gb(1.7)                        # 1.7B teacher bf16 weight bytes = 3.4 GB
N_PROMPTS_, NEW = 2048, 256
GEN_TOKENS = N_PROMPTS_ * NEW               # 0.52 M generated tokens
SEQ_LEN, DECODE_BATCH, PREFILL_BATCH = 384, 16, 16
STEPS_BASE = 1500                           # Part C's matched student steps

# teacher traffic per arm, in GB
gb_decode = GEN_TOKENS / DECODE_BATCH * T_GB              # one weight read per 16 tokens
gb_prefill = (N_PROMPTS_ / PREFILL_BATCH) * T_GB          # one weight read per batch
teacher_gb = {"cached": gb_prefill, "seqkd": gb_decode, "trace": 0.0}

# student traffic: ~3 passes over the 16-byte full fine-tune working set per step
STUDENT_STEP_GB = 3 * full_ft_gb(0.36)                    # 17.3 GB per optimizer step
student_gb = STEPS_BASE * STUDENT_STEP_GB

sec = lambda gb: gb / BW
total_s = {a: sec(teacher_gb[a]) + sec(student_gb) for a in teacher_gb}
budget = max(total_s.values())

print(f"{'arm':>8} {'teacher s':>10} {'student s':>10} {'total s':>9} "
      f"{'extra s':>9} {'steps at matched compute':>25}")
extra_steps = {}
for a in ("cached", "seqkd", "trace"):
    extra = budget - total_s[a]
    extra_steps[a] = int(extra / sec(STUDENT_STEP_GB))
    print(f"{a:>8} {sec(teacher_gb[a]):>10.1f} {sec(student_gb):>10.1f} "
          f"{total_s[a]:>9.1f} {extra:>9.1f} {STEPS_BASE + extra_steps[a]:>25,d}")

# identity 1: seconds really are bytes over bandwidth
assert abs(sec(gb_decode) - gb_decode / BW) < 1e-9
# identity 2: after reallocating the slack into steps, every arm spends the budget
for a in extra_steps:
    spent = sec(teacher_gb[a]) + sec(student_gb) + extra_steps[a] * sec(STUDENT_STEP_GB)
    assert abs(spent - budget) <= sec(STUDENT_STEP_GB), f"{a} does not exhaust the budget"
# identity 3: the ordering the economics force
assert extra_steps["trace"] > extra_steps["cached"] > extra_steps["seqkd"] == 0
assert total_s["seqkd"] == budget, "teacher decode must be the binding cost"

# The flip question, made concrete with an ILLUSTRATIVE quality model (labeled as
# such): agreement ~ a + b*log2(steps/1500), with offsets from Part C's ranges
# (cached +3 points over seqkd at equal steps, trace -2 for domain shift) and a
# deliberately modest b of 1.5 points per doubling of steps.
a0 = {"cached": 63.0, "seqkd": 60.0, "trace": 58.0}
b = 1.5
q = {a: a0[a] + b * math.log2((STEPS_BASE + extra_steps[a]) / STEPS_BASE)
     for a in a0}
print("\nillustrative matched-compute quality (NOT a measurement):")
for a in sorted(q, key=q.get, reverse=True):
    print(f"  {a:>8}: {q[a]:5.1f} (was {a0[a]:.1f} at matched steps)")
assert q["trace"] > a0["trace"] and q["cached"] > a0["cached"]
print("\naccounting identities hold; the slack is where the ranking can flip")

     arm  teacher s  student s   total s   extra s  steps at matched compute
  cached        1.6       94.9      96.5     406.5                     7,922
   seqkd      408.1       94.9     503.0       0.0                     1,500
   trace        0.0       94.9      94.9     408.1                     7,947

illustrative matched-compute quality (NOT a measurement):
    cached:  66.6 (was 63.0 at matched steps)
     trace:  61.6 (was 58.0 at matched steps)
     seqkd:  60.0 (was 60.0 at matched steps)

accounting identities hold; the slack is where the ranking can flip


**Interpretation.** The live table is the exercise's answer. At matched student steps the
seqkd arm quietly received the largest total budget, because its teacher-decode phase alone
(about 110,000 GB of weight traffic even batched 16 wide, which is roughly 400 seconds at
273 GB/s) dwarfs the cached arm's prefill traffic (a few hundred GB, under two seconds) and
the trace arm's zero. Matching total compute hands that slack back as extra student steps:
the asserted ordering shows trace gets the most extra steps, cached next, seqkd none, and
with this corpus and batch the cheaper arms afford several times the baseline step count.

Does the ranking change? The arithmetic says it *can*, and shows exactly where. The final
block is an illustrative model, labeled as such and not a measurement: it takes Part C's
expected offsets at matched steps (cached ahead of seqkd by a few points, trace behind for
domain shift) and a modest 1.5 points per doubling of steps. Under those numbers cached
extends its lead and trace overtakes seqkd outright (61.6 against 60.0 in the printed
table), because the arms that skipped the expensive decode phase convert the savings into
multiple doublings of student steps. What would count as
a real flip when the gated runs of Part C are rerun this way: trace-sft overtaking seqkd on
agreement, which the extra-step arithmetic makes plausible whenever the per-doubling gain
exceeds about half the domain-shift penalty. What would refute the referee's premise:
quality curves already flat at 1500 steps (the extra steps buy nothing, so the matched-step
table stands). That is why this comparison is the one the SeqKD papers rarely show: the
mode-matched corpus looks best exactly when the referee ignores who paid for it. One
defensible verdict, not the only one: on this hardware class, at fixed total compute, cached
logits when you have white-box access, traces when you have none, and teacher-decode SeqKD
only when someone else already paid the decode bill.